In [2]:
from pathlib import Path
import numpy as np
import tensorflow as tf

MODEL_PATH, IMAGE_PATH = Path("efficientnet_plants.h5"), Path("128.jpg")
DATA_DIR = Path(r"C:\Users\harsh\OneDrive\Desktop\plant class\without background class")
CLASS_NAMES = None  # Or set class folder names in alphabetical order.

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")
if CLASS_NAMES is None and not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Set DATA_DIR or CLASS_NAMES; class folder not found: {DATA_DIR}")

class_names = list(CLASS_NAMES) if CLASS_NAMES is not None else sorted(
    folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()
)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
if len(class_names) != model.output_shape[-1]:
    raise ValueError(f"Found {len(class_names)} labels for {model.output_shape[-1]} model outputs")
print(f"Loaded model with {len(class_names)} classes.")

FileNotFoundError: Model file not found: efficientnet_plants.h5

In [ ]:
def predict_image(image_path: str | Path, top_k: int = 3) -> list[dict[str, str | float]]:
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image file not found: {image_path}")

    image = tf.keras.utils.load_img(image_path, target_size=(256, 256), color_mode="rgb")
    batch = tf.keras.applications.efficientnet.preprocess_input(
        np.expand_dims(tf.keras.utils.img_to_array(image), axis=0)
    )
    probabilities = np.asarray(model.predict(batch, verbose=0))[0]
    indices = np.argsort(probabilities)[::-1][:min(max(int(top_k), 1), len(class_names))]
    results = [{"class": class_names[i], "confidence": float(probabilities[i])} for i in indices]

    print(f"Predicted plant: {results[0]['class']} ({results[0]['confidence']:.2%})")
    for result in results[1:]:
        print(f"Other: {result['class']} ({result['confidence']:.2%})")
    return results

In [3]:
result = predict_image(IMAGE_PATH)
result

Predicted plant: Insulin (47.74%)
Other: Catharanthus (8.70%)
Other: Arive Dantu (5.54%)


[{'class': 'Insulin', 'confidence': 0.4773635268211365},
 {'class': 'Catharanthus', 'confidence': 0.08696568757295609},
 {'class': 'Arive Dantu', 'confidence': 0.055391255766153336}]